# Notebook 7 — Gameplay Video Generation

Records `.mp4` gameplay videos from early, mid, and final checkpoints using `imageio`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
    !pip install -q imageio imageio-ffmpeg
except ImportError:
    SAVE_DIR = 'models'

import os, glob
import numpy as np
import imageio
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3 import DQN, A2C, PPO

gym.register_envs(ale_py)

VIDEO_DIR = os.path.join(SAVE_DIR, 'videos')
os.makedirs(VIDEO_DIR, exist_ok=True)

ALGO_MAP = {'dqn': DQN, 'a2c': A2C, 'ppo': PPO}

In [ ]:
def record_video(algo_name, checkpoint_path, env_id, output_path, fps=30):
    AlgoClass = ALGO_MAP[algo_name]
    env = gym.make(env_id, render_mode='rgb_array')
    env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=True)
    env = FrameStackObservation(env, stack_size=4)

    model = AlgoClass.load(checkpoint_path)
    frames = []
    obs, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    env.close()
    imageio.mimwrite(output_path, frames, fps=fps)
    print(f'  Saved: {output_path}  (reward={total_reward:.1f}, frames={len(frames)})')
    return total_reward

In [ ]:
# Record early / mid / final checkpoints for DQN Pong
RUN_DIR = os.path.join(SAVE_DIR, 'dqn_pong_default')
ENV_ID  = 'ALE/Pong-v5'
ALGO    = 'dqn'

all_ckpts = sorted(glob.glob(os.path.join(RUN_DIR, 'dqn_*_steps.zip')))

# Pick early (~100k), mid (~500k or 1M), and final
if len(all_ckpts) >= 3:
    selected = {
        'early': all_ckpts[0],
        'mid':   all_ckpts[len(all_ckpts)//2],
        'final': all_ckpts[-1],
    }
    for label, ckpt in selected.items():
        print(f'Recording {label}: {os.path.basename(ckpt)}')
        out = os.path.join(VIDEO_DIR, f'{ALGO}_pong_{label}.mp4')
        record_video(ALGO, ckpt, ENV_ID, out)
else:
    print('Not enough checkpoints yet — run Notebook 2 first.')

In [ ]:
# Preview a video inline (Colab only)
try:
    from IPython.display import HTML
    from base64 import b64encode
    video_path = os.path.join(VIDEO_DIR, f'{ALGO}_pong_final.mp4')
    with open(video_path, 'rb') as f:
        video_data = f.read()
    b64 = b64encode(video_data).decode()
    display(HTML(f'<video controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
except Exception as e:
    print(f'Preview not available: {e}')